# Silver to Gold Layer

## Purpose
This notebook reads incremental transaction data from the Silver layer and generates daily business KPIs.

### Source
retailanalytics.silver.silver_transaction

### Target
retailanalytics.gold.daily_sales

### KPIs
- Total Revenue
- Completed Transactions
- Failed Transactions
- Pending Transactions
- Total Units Sold

### Processing Pattern
Structured Streaming + foreachBatch + Delta MERGE

In [0]:
from pyspark.sql import functions as F

In [0]:
gold_checkpoint = (
    "/Volumes/retailanalytics/secrets/"
    "kafkacerts/checkpoints/gold_daily_sales"
)

In [0]:
silver_stream = (
    spark.readStream.table(
        "retailanalytics.silver.silver_transaction"
    )
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retailanalytics.gold.daily_sales
(
    sale_date DATE,
    total_revenue DOUBLE,
    completed_transactions BIGINT,
    failed_transactions BIGINT,
    pending_transactions BIGINT,
    total_units BIGINT
)
USING DELTA;

In [0]:
def upsert_daily_sales(batch_df, batch_id):

    daily_batch = (
        batch_df
        .groupBy(
            F.to_date("transaction_ts").alias("sale_date")
        )
        .agg(
            F.sum(
                F.when(
                    F.col("transaction_status") == "COMPLETED",
                    F.col("gross_amount")
                ).otherwise(0)
            ).alias("total_revenue"),

            F.sum(
                F.when(
                    F.col("transaction_status") == "COMPLETED",
                    1
                ).otherwise(0)
            ).alias("completed_transactions"),

            F.sum(
                F.when(
                    F.col("transaction_status") == "FAILED",
                    1
                ).otherwise(0)
            ).alias("failed_transactions"),

            F.sum(
                F.when(
                    F.col("transaction_status") == "PENDING",
                    1
                ).otherwise(0)
            ).alias("pending_transactions"),

            F.sum(
                F.when(
                    F.col("transaction_status") == "COMPLETED",
                    F.col("quantity")
                ).otherwise(0)
            ).alias("total_units")
        )
    )

    daily_batch.createOrReplaceTempView(
        "daily_sales_updates"
    )

    spark.sql("""
        MERGE INTO retailanalytics.gold.daily_sales AS target
        USING daily_sales_updates AS source
        ON target.sale_date = source.sale_date

        WHEN MATCHED THEN
          UPDATE SET
            target.total_revenue =
                target.total_revenue + source.total_revenue,

            target.completed_transactions =
                target.completed_transactions + source.completed_transactions,

            target.failed_transactions =
                target.failed_transactions + source.failed_transactions,

            target.pending_transactions =
                target.pending_transactions + source.pending_transactions,

            target.total_units =
                target.total_units + source.total_units

        WHEN NOT MATCHED THEN
          INSERT (
              sale_date,
              total_revenue,
              completed_transactions,
              failed_transactions,
              pending_transactions,
              total_units
          )
          VALUES (
              source.sale_date,
              source.total_revenue,
              source.completed_transactions,
              source.failed_transactions,
              source.pending_transactions,
              source.total_units
          )
    """)

In [0]:
(
    silver_stream.writeStream
        .foreachBatch(upsert_daily_sales)
        .option(
            "checkpointLocation",
            gold_checkpoint
        )
        .trigger(availableNow=True)
        .start()
)

In [0]:
%sql
SELECT *
FROM retailanalytics.gold.daily_sales
ORDER BY sale_date DESC;